# Project 1: Production Batch ETL Pipeline (CSV/JSON to Partitioned Parquet Lake)

**Module**: 13_end_to_end_projects  
**Notebook**: 01_batch_etl_pipeline.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Building a full end-to-end production batch ETL pipeline: schema validation, extraction from CSV & JSON, deduplication, enrichment joins, metric aggregation, and writing to a partitioned Parquet data lake.
- **Why does this topic exist?**: Integrates all core skills (schemas, cleaning, joins, partitioning, and writing) into an industry-standard Data Engineering project.
- **What problem does it solve?**: Raw data arriving in disparate formats with missing values, dirty strings, and unindexed tables.
- **Where is it used in real-world Data Engineering?**: Core responsibility of every Data Engineer in modern enterprise data teams.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 4–6 h |
| **Prerequisites** | DataFrames, data sources, partitioning, writes |
| **Required depth** | Working knowledge: can build and defend a production batch ETL design |
| **Data Engineer** | Implement an idempotent, partition-aware batch pipeline. |
| **Senior Data Engineer** | Design ingestion, validation, and publication layers with clear contracts. |
| **Spark specialist** | Reason about failure recovery, backfills, and table layout. |
| **Next topic** | `13_end_to_end_projects/02_large_scale_sales_analytics.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

A batch ETL pipeline is more than a sequence of reads and writes. It needs a landing contract, schema
validation, deterministic transformation, partition-aware output, and a recovery plan. The project ties together
the earlier modules: file formats, joins, partitioning, writes, and performance diagnosis in one pipeline.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Landing zone** | Raw input area | Immutable source files with a manifest or batch ID |
| **Staging** | Validated intermediate data | Cleaned and typed records |
| **Curated layer** | Published output | Partitioned, query-ready tables |
| **Idempotency** | Safe reruns | Same batch produces one output copy |
| **Data contract** | Expected schema and quality | Enforced at ingestion |
| **Backfill** | Reprocess history | Partition-scoped and auditable |

### How it works, step by step

1. Read the batch manifest and discover input files.
2. Validate schema, row counts, and key constraints.
3. Clean and conform types, deduplicate, and enrich.
4. Write to a staging location partitioned by batch/date.
5. Validate the staged output against quality rules.
6. Publish to the curated layer with dynamic partition overwrite or a merge.
7. Record lineage and metrics for the run.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Project Architecture Diagram
```
[Raw CSV: orders.csv] --------+
                                |
[Raw CSV: customers.csv] ------> [Extract & Validate Schemas]
                                |                |
[Raw JSON: order_events.json]-+                v
                                    [Clean, Deduplicate & Impute Nulls]
                                                 |
                                                 v
                                    [Broadcast & Star-Schema Joins]
                                                 |
                                                 v
                                    [Derived Revenue Metrics & Aggs]
                                                 |
                                                 v
                                    [Load: Partitioned Parquet Lake]
                                    (year=YYYY / month=MM / status=...)
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import time

spark = (
    SparkSession.builder
    .appName("Batch_ETL_Project")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Step 1: Explicit Schema Extraction

In [ ]:
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("order_ts", TimestampType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("unit_price", DoubleType(), False),
    StructField("discount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("channel", StringType(), True)
])

orders_raw = spark.read.option("header", True).schema(orders_schema).csv("/opt/spark-data/retail/orders.csv")
customers_raw = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/customers.csv")
products_raw = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/products.csv")

print(f"Extracted: {orders_raw.count():,} orders, {customers_raw.count():,} customers, {products_raw.count():,} products.")

## 4. Step 2: Data Cleaning, Deduplication, and Normalization

In [ ]:
# Deduplicate and handle nulls
orders_clean = (
    orders_raw
    .dropDuplicates(["order_id"])
    .filter(F.col("quantity") > 0)
    .filter(F.col("unit_price") > 0.0)
    .na.fill({"discount": 0.0, "status": "Pending", "channel": "Unknown"})
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .withColumn("gross_rev", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("net_rev", F.round(F.col("gross_rev") * (1.0 - F.col("discount")), 2))
)

print(f"Cleaned orders count: {orders_clean.count():,}")
orders_clean.show(3)

## 5. Step 3: Star-Schema Broadcast Enrichment Joins

In [ ]:
# Broadcast small dimension tables (products and customers)
enriched = (
    orders_clean
    .join(F.broadcast(products_raw.select("product_id", "product_name", "category")), on="product_id", how="left")
    .join(F.broadcast(customers_raw.select("customer_id", "country", "region", "segment", "loyalty_tier")), on="customer_id", how="left")
)

enriched.show(3)

## 6. Step 4: Loading into Partitioned Parquet Lakehouse

In [ ]:
lake_output = "/opt/spark-data/warehouse/project_batch_lake"

(
    enriched
    .coalesce(4)
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(lake_output)
)

print("ETL Batch Pipeline successfully loaded into data lake!")
# Verify roundtrip read
read_back = spark.read.parquet(lake_output)
print(f"Verified rows written: {read_back.count():,}")

---

## Experiment: Verifying Partition Directory Structure on Disk

- **1. What are we testing?**: Inspect the generated file tree on disk to confirm year=YYYY/month=MM partitioning.
- **2. What do we expect?**: Nested year and month subdirectories containing Parquet files.

In [ ]:
dirs = [d for d in os.listdir(lake_output) if d.startswith("year=")]
print("Root partition folders in data lake:", dirs[:4])

**3. What happened?**: Parquet files were successfully segregated into year and month directories.  
**4. Why did it happen?**: partitionBy('year','month') configured the directory layout.  
**5. Practical Engineering Lesson**: Partitioned data lakes allow downstream consumers to query specific time slices with zero full scans.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Retail order ingestion

- **Industry**: Retail
- **Scenario**: Daily order files arrive from an e-commerce platform and must be published for finance and operations.
- **Problem**: The pipeline must handle late files, duplicate orders, and reruns without corrupting the curated table.
- **Why this concept is relevant**: A manifest-driven ingestion with validation and partition-scoped publication makes retries safe.
- **Implementation idea**: Validate the batch, deduplicate by order ID, write to staging, then publish by date partition.
- **Expected result**: A reliable daily table with one row per order.
- **Alternative**: Use a managed ingestion service.
- **Trade-offs**: A managed service reduces operations but may not support custom validation and backfills.

### Use Case 2: Partner feed onboarding

- **Industry**: Logistics
- **Scenario**: A new partner sends shipment events in a different schema and cadence.
- **Problem**: The pipeline must onboard the feed without breaking existing consumers.
- **Why this concept is relevant**: A landing contract and a schema-validation gate isolate the new feed and surface mismatches early.
- **Implementation idea**: Land raw files, validate against the contract, quarantine bad records, and publish a conformed view.
- **Expected result**: A new feed integrated with measurable quality.
- **Alternative**: Ask the partner to change formats.
- **Trade-offs**: A format change may be cleaner but is often outside the team's control.

### Use Case 3: Historical backfill

- **Industry**: Financial services
- **Scenario**: A regulatory change requires reprocessing two years of transactions.
- **Problem**: A full overwrite would block readers and risk deleting valid data.
- **Why this concept is relevant**: Partition-scoped backfills with staging and validation allow incremental replacement.
- **Implementation idea**: Process one month at a time, validate counts, and overwrite only the affected partitions.
- **Expected result**: A complete backfill with no reader downtime.
- **Alternative**: Build a parallel table and swap it.
- **Trade-offs**: A parallel table isolates the backfill but doubles storage during the migration.

---

## Business Value

- Reliable daily data for downstream teams.
- Safe reruns and backfills without manual cleanup.
- Measurable data-quality gates.
- Lower operational cost through automation.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Make every write idempotent and partition-scoped.
- Validate before publishing, not after.
- Record batch IDs and row counts for audit.
- Use dynamic partition overwrite for backfills.
- Test failure and rerun behavior, not only the happy path.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Run the pipeline twice

**Steps**

1. Run the ETL for one batch and record the output key count.
2. Rerun the same batch without changing the input.
3. Verify that the output contains exactly one copy of each key.

**What to observe**: An idempotent design produces the same output after a rerun; a naive append duplicates rows.

**Healthy result**: Key counts are stable across reruns.

**Problematic result**: The curated table contains duplicates or partial files.

**Engineering takeaway**: Idempotency is the core requirement of a production ETL.

---

## Trade-offs and Alternatives

### Advantages

- Reuses Spark skills
- Scales to large batches
- Flexible formats and sinks

### Limitations

- Operational complexity
- File sinks are not transactional by default
- Backfills require care

### When to use it

- Daily ingestion
- Data lake curation
- Partner onboarding

### When not to use it

- Blind overwrites
- Unvalidated publication
- Non-idempotent appends

### Alternatives inside Spark

- DataFrameReader/Writer
- Partitioning
- Validation

### Alternatives outside Spark

- Managed ETL services
- Transactional table formats
- Workflow orchestrators

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Calculate Aggregated Regional Revenue
**Problem Statement**: Using read_back, compute total net revenue by region.  
**Input / Context**: read_back

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Verify No Null Keys
**Problem Statement**: Assert that no order_id or customer_id is null in the lake output.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Incremental Ingestion Pattern
**Problem Statement**: Explain how to run this job incrementally with Dynamic Partition Overwrite.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The backfill that broke the dashboard

**Scenario**: A backfill replaces a month of data, but the dashboard shows missing dates.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the partition overwrite mode and the write path.
2. Verify that the staging data contains every expected date.
3. Compare partition lists before and after the backfill.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Calculate Aggregated Regional Revenue):
```python
read_back.groupBy('region').agg(F.round(F.sum('net_rev'), 2).alias('total')).show()
```

#### Solution 2 (Verify No Null Keys):
```python
assert read_back.filter('order_id IS NULL OR customer_id IS NULL').count() == 0; print('Assertion passed!')
```

#### Solution 3 (Incremental Ingestion Pattern):
```python
print('Set partitionOverwriteMode=dynamic to overwrite only the specific year/month partitions in the incoming batch.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **ETL Blueprint**: Extract with schema -> Clean/Deduplicate -> Broadcast Join -> Aggregate -> Partitioned Write.
- **Production Parquet**: Columnar format with Snappy compression and partitioned directories.

### Essential Technical Terms
- `ETL`: Extract, Transform, Load.
- `Star-Schema`: Fact table surrounded by dimension tables.

### Common Engineering Mistakes
- **Omitting deduplication**: Causes duplicate analytics and inaccurate financial reporting.
- **Writing unpartitioned outputs**: Slows downstream queries.

### Interview Questions on This Topic
- **Q: What is the advantage of broadcast joins in an ETL pipeline?**
  - *A*: They eliminate network shuffles of the massive fact table by distributing the dimension table to every executor once.


### More Interview Questions

- **Q: How do you make a batch ETL idempotent?**
  - *A*: Use a deterministic batch key and a partition-scoped overwrite or merge so rerunning the same batch replaces rather than duplicates its output.
- **Q: Why validate before publishing?**
  - *A*: Publishing bad data can break downstream consumers and is harder to undo than stopping the batch in staging.

### What to Study Next
Open 02_large_scale_sales_analytics.ipynb for advanced analytical windowing.

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')